In [15]:
from netgen.occ import *
import ngsolve
from ngsolve import *
from ngsolve.webgui import Draw
import numpy as np

In [16]:
box = Box((0,0,0), (3,0.6,1))
box.faces.name="outer"
cyl = sum( [Cylinder((0.5+i,0,0.5), Y, 0.25,0.8) for i in range(3)] )
cyl.faces.name="cyl"
geo = box-cyl


Draw(geo, height="3vh")

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'ngsolve_version': 'Netgen x.x', 'mesh_dim': 3,…

BaseWebGuiScene

In [17]:
cylboxedges = geo.faces["outer"].edges * geo.faces["cyl"].edges
cylboxedges.name = "cylbox"
geo = geo.MakeChamfer(cylboxedges, 0.03)

In [18]:
geo.faces.Min(X).name = "fix"
geo.faces.Max(X).name = "force"

Draw(geo, height="3vh");

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'ngsolve_version': 'Netgen x.x', 'mesh_dim': 3,…

In [19]:
from ngsolve import *
from ngsolve.webgui import Draw
mesh = Mesh(OCCGeometry(geo).GenerateMesh(maxh=0.1)).Curve(3)
Draw(mesh, height="3vh");

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…

In [20]:
dt = 0.1
E, nu = 210, 0.2
mu  = E / 2 / (1+nu)
lam = E * nu / ((1+nu)*(1-2*nu))

def Stress(strain):
    return 2*mu*strain + lam*Trace(strain)*Id(3)

In [21]:
t = Parameter(0.0)
fes = VectorH1(mesh, order=3, dirichlet="fix")
u,v = fes.TnT()
gfu = GridFunction(fes)

with TaskManager():
    m_int = u*v*dx
    m = BilinearForm(m_int)
    m.Assemble()
    a_int = InnerProduct(Stress(Sym(Grad(u))), Sym(Grad(v))).Compile()*dx
    a = BilinearForm(a_int)
    a.Assemble()
    mstar = BilinearForm(fes)
    mstar += m_int
    mstar += dt*a_int
    pre = Preconditioner(mstar, "bddc")
    mstar.Assemble()
    

In [22]:
force = CF( (20,0,0))*cos(2*np.pi*t) 
f = LinearForm(force*v*ds("force"))

In [23]:
from ngsolve.krylovspace import CGSolver

inv = CGSolver(mstar.mat, pre, printrates=True, tol=1e-8)

In [24]:
gfut = GridFunction(gfu.space,multidim=0)
tend = t.Get()+1
while t.Get() <= tend:
    f.Assemble()

    rhs = dt * f.vec + m.mat * gfu.vec
    gfu.vec.data = inv * rhs
    gfut.AddMultiDimComponent(gfu.vec)
    
    t.Set(t.Get()+dt)
    

CG iteration 1, residual = 0.9677622196265393     
CG iteration 2, residual = 0.4216504030973621     
CG iteration 3, residual = 0.49506863381129074     
CG iteration 4, residual = 0.36444227095796294     
CG iteration 5, residual = 0.33823095664858527     
CG iteration 6, residual = 0.31144780646083026     
CG iteration 7, residual = 0.20728155875099066     
CG iteration 8, residual = 0.1651857041032265     
CG iteration 9, residual = 0.12176440374275595     
CG iteration 10, residual = 0.09838535066857836     
CG iteration 11, residual = 0.08076735123439664     
CG iteration 12, residual = 0.054534963996396915     
CG iteration 13, residual = 0.04214223587025113     
CG iteration 14, residual = 0.03288356962152987     
CG iteration 15, residual = 0.022324972879617083     
CG iteration 16, residual = 0.017980961401755303     
CG iteration 17, residual = 0.01463366619559023     
CG iteration 18, residual = 0.010949397583572816     
CG iteration 19, residual = 0.008044677759683197     


In [25]:
with TaskManager():
    fesstress = MatrixValued(H1(mesh,order=3), symmetric=True)
    gfstress = GridFunction(fesstress)
    gfstress.Interpolate (Stress(Sym(Grad(gfu))))

In [26]:
Draw (gfut, mesh, height="2vh", deformation=gfut, interpolate_multidim=True, animate=True)

WebGuiWidget(layout=Layout(height='2vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…

BaseWebGuiScene

In [27]:
Draw (Norm(gfstress), mesh, deformation=1e4*gfu, draw_vol=False, order=3, height="2vh");

WebGuiWidget(layout=Layout(height='2vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…